# Lab 6 - Task 4: Unsharp Masking — Parameter Exploration

Explore the impact of `filt_size` and `k` on image sharpness using the custom convolution-based unsharp masking approach.

**Formula:** `image_sharp = image_input + k * (image_input - image_blur)`

- `filt_size` controls how much the image is blurred (larger = more blur = edges captured at coarser scale)
- `k` controls the sharpening strength and direction

In [ ]:
import copy
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Helper Functions

In [ ]:
def box_filt(n):
    if n <= 0 or n % 2 == 0:
        raise ValueError("Window size must be a positive odd integer.")
    return np.ones((n, n), np.float32) / (n * n)


def apply_filters(image_input, box, filt_size):
    pad_size = int(np.ceil(filt_size / 2))
    image_padded = np.pad(
        image_input,
        pad_width=((pad_size, pad_size), (pad_size, pad_size)),
        mode='symmetric'
    )
    image_box = copy.deepcopy(image_input)
    row, column = image_input.shape
    for i in range(row):
        for j in range(column):
            patch_curr = image_padded[i:i + filt_size, j:j + filt_size]
            image_box[i, j] = np.sum(box * patch_curr)
    return image_box


def unsharp_mask(image_float, filt_size, k):
    """Apply unsharp masking: image + k*(image - blurred)."""
    box = box_filt(filt_size)
    image_blur = apply_filters(image_float, box, filt_size)
    image_diff = image_float - image_blur
    image_sharp = image_float + k * image_diff
    return np.clip(image_sharp, 0, 1)

## Load Image

In [ ]:
image_raw = cv2.imread('image.png', 0)

if image_raw is None:
    image_raw = np.random.randint(50, 200, (128, 128), dtype=np.uint8)
    print("No image file found — using a synthetic test image.")

image_input = image_raw.astype('float32') / 255.0
print(f"Image shape: {image_input.shape}")

## Part 4A: Vary filt_size (k fixed at 2)

In [ ]:
filt_sizes = [3, 5, 7, 9, 11, 15]
k_fixed = 2

fig, axes = plt.subplots(2, len(filt_sizes) // 2 + 1, figsize=(18, 8))
axes = axes.ravel()

axes[0].imshow(image_input, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for idx, fs in enumerate(filt_sizes):
    result = unsharp_mask(image_input, fs, k_fixed)
    axes[idx + 1].imshow(result, cmap='gray')
    axes[idx + 1].set_title(f'filt_size={fs}, k={k_fixed}')
    axes[idx + 1].axis('off')

# Hide any unused subplots
for ax in axes[len(filt_sizes) + 1:]:
    ax.set_visible(False)

plt.suptitle(f'Varying filt_size (k={k_fixed} fixed)', fontsize=13)
plt.tight_layout()
plt.show()

## Part 4B: Vary k (filt_size fixed at 5)

In [ ]:
k_values = [-2, -1, -0.5, 0, 0.5, 1, 2, 3, 5, 9]
filt_fixed = 5

ncols = 5
nrows = int(np.ceil((len(k_values) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4))
axes = axes.ravel()

axes[0].imshow(image_input, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for idx, k in enumerate(k_values):
    result = unsharp_mask(image_input, filt_fixed, k)
    axes[idx + 1].imshow(result, cmap='gray')
    axes[idx + 1].set_title(f'k={k}, filt_size={filt_fixed}')
    axes[idx + 1].axis('off')

for ax in axes[len(k_values) + 1:]:
    ax.set_visible(False)

plt.suptitle(f'Varying k (filt_size={filt_fixed} fixed)', fontsize=13)
plt.tight_layout()
plt.show()

## Part 4C: Combined Grid — filt_size × k

In [ ]:
filt_sizes_grid = [3, 7, 15]
k_grid = [-1, 0, 1, 3, 9]

fig, axes = plt.subplots(len(filt_sizes_grid), len(k_grid), figsize=(20, 12))

for row_idx, fs in enumerate(filt_sizes_grid):
    for col_idx, k in enumerate(k_grid):
        result = unsharp_mask(image_input, fs, k)
        axes[row_idx][col_idx].imshow(result, cmap='gray')
        axes[row_idx][col_idx].set_title(f'fs={fs}, k={k}', fontsize=8)
        axes[row_idx][col_idx].axis('off')

plt.suptitle('Combined: filt_size (rows) × k (columns)', fontsize=13)
plt.tight_layout()
plt.show()

## Analysis: Impact of filt_size and k

| Parameter | Value | Effect |
|---|---|---|
| `filt_size` | Small (3) | Only fine/high-frequency edges are enhanced |
| `filt_size` | Large (15) | Broader/coarser edges and structures are amplified |
| `k` | Negative (< 0) | Image is **smoothed/blurred** instead of sharpened |
| `k` | 0 | No change — returns original |
| `k` | 0–1 | Mild sharpening |
| `k` | 1–3 | Strong sharpening, edges clearly enhanced |
| `k` | > 3 | Over-sharpening — introduces artifacts and halos |
| `k` | 9 | Extreme sharpening — mostly noise and edge ringing visible |

**Summary:**
- `filt_size` determines the **scale** of features that are sharpened (fine vs. coarse).
- `k` controls the **intensity** and **direction** of the effect: positive values sharpen, negative values smooth.
- Optimal results typically use moderate values: `filt_size` ∈ {5, 7} and `k` ∈ {1, 2}.